<a href="https://colab.research.google.com/github/vidya-kt/RetroRebirth/blob/main/RetroRebirth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- BLOCK 1: INSTALL, SETUP & DATA GENERATION ---

!pip install -q torch torchvision
!pip install -q modelscope==1.9.5

import os
import cv2
import numpy as np
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# 1. SETUP PATHS
base_path = Path('/content/retro_dataset')
path_A = base_path/'negatives'
path_B = base_path/'positives'
path_A.mkdir(parents=True, exist_ok=True)
path_B.mkdir(parents=True, exist_ok=True)

# 2. CONFIGURATION
NUM_IMAGES = 2200
IMG_SIZE = 256
ORANGE_MASK = [70, 150, 220]  # BGR

print(f"Starting generation of {NUM_IMAGES} synthetic training pairs...")

urls = [f"https://loremflickr.com/512/512/portrait,landscape,city,vintage?random={i}"
        for i in range(NUM_IMAGES)]

def generate_pair(idx):
    try:
        resp = requests.get(urls[idx], timeout=10)
        if resp.status_code != 200:
            return

        arr = np.asarray(bytearray(resp.content), dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is None:
            return

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # Save Positive
        cv2.imwrite(str(path_B/f'{idx}.jpg'), img)

        # Create Synthetic Negative
        inverted = 255 - img
        norm = inverted / 255.0
        mask = np.full_like(norm, ORANGE_MASK) / 255.0
        neg = (norm * mask * 255) + np.random.normal(0, 10, mask.shape)
        neg = np.clip(neg, 0, 255).astype(np.uint8)

        cv2.imwrite(str(path_A/f'{idx}.jpg'), neg)

    except Exception:
        pass

# Multithreading for fast generation
with ThreadPoolExecutor(max_workers=16) as ex:
    ex.map(generate_pair, range(NUM_IMAGES))

print(f"Data Generation Complete. Saved to {base_path}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 21.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.6/485.6 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 11.5 MB/s eta 0:00:

In [ ]:
# --- BLOCK 2: MODEL DEFINITION & TRAINING ---

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image as PILImage

# Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 1. U-NET MODEL
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self.conv_block(3, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.bottleneck = self.conv_block(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)

        self.final = nn.Conv2d(64, 3, 1)

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(nn.MaxPool2d(2)(e1))
        e3 = self.enc3(nn.MaxPool2d(2)(e2))
        b = self.bottleneck(nn.MaxPool2d(2)(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return torch.sigmoid(self.final(d1))

# 2. DATASET
class RetroDataset(Dataset):
    def __init__(self, root_dir):
        self.neg_files = sorted(list((root_dir/'negatives').glob('*.jpg')))
        self.pos_dir = root_dir/'positives'
        self.transform = transforms.Compose([transforms.ToTensor()])

    def __len__(self):
        return len(self.neg_files)

    def __getitem__(self, idx):
        n_path = self.neg_files[idx]
        p_path = self.pos_dir / n_path.name

        try:
            neg = PILImage.open(n_path).convert("RGB")
            pos = PILImage.open(p_path).convert("RGB")
            return self.transform(neg), self.transform(pos)
        except:
            return torch.zeros(3,256,256), torch.zeros(3,256,256)

# 3. TRAINING
model = SimpleUNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

dataloader = DataLoader(RetroDataset(base_path), batch_size=16, shuffle=True)

print("Training Negative Converter...")
EPOCHS = 8
for epoch in range(EPOCHS):
    epoch_loss = 0
    for neg, pos in dataloader:
        neg, pos = neg.to(device), pos.to(device)
        optimizer.zero_grad()
        loss = criterion(model(neg), pos)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(dataloader):.4f}")

print("Model Training Complete.")

# --- SAVE MODEL ---
MODEL_PATH = "/content/retro_rebirth_unet.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model Saved at: {MODEL_PATH}")


⚙️ Training on device: cuda
🔥 Training Negative Converter...
Epoch 1/8 | Loss: 0.0051
Epoch 2/8 | Loss: 0.0027
Epoch 3/8 | Loss: 0.0030
Epoch 4/8 | Loss: 0.0027
Epoch 5/8 | Loss: 0.0025
Epoch 6/8 | Loss: 0.0026
Epoch 7/8 | Loss: 0.0025
Epoch 8/8 | Loss: 0.0024
✅ Model Training Complete.
💾 Model Saved at: /content/retro_rebirth_unet.pth


In [ ]:
# --- BLOCK 3: IMPLEMENTATION & UI ---
import ipywidgets as widgets
from IPython.display import display
import io
import matplotlib.pyplot as plt
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
import cv2
import numpy as np
from PIL import Image as PILImage
from torchvision import transforms
import torch
import torch.nn as nn
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------
# REDEFINE EXACT SAME U-NET USED IN TRAINING
# ---------------------------------------------------
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self.conv_block(3, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.bottleneck = self.conv_block(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        self.final = nn.Conv2d(64, 3, 1)

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(nn.MaxPool2d(2)(e1))
        e3 = self.enc3(nn.MaxPool2d(2)(e2))
        b  = self.bottleneck(nn.MaxPool2d(2)(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.final(d1))

# ---------------------------------------------------
# LOAD TRAINED MODEL WEIGHTS
# ---------------------------------------------------
MODEL_PATH = "/content/retro_rebirth_unet.pth"
model = SimpleUNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Loaded trained RetroRebirth U-Net successfully!")

# ---------------------------------------------------
# LOAD DDColor
# ---------------------------------------------------
print("Loading DDColor...")
color_pipeline = pipeline(Tasks.image_colorization, model='damo/cv_ddcolor_image-colorization')
print("DDColor Ready!")

# ---------------------------------------------------
# UI SETUP
# ---------------------------------------------------
btn_upload = widgets.FileUpload(accept='image/*', multiple=False)
dropdown_mode = widgets.Dropdown(options=['Negative', 'B&W'], value='Negative', description='Mode:')
btn_run = widgets.Button(description="Rebirth Image", button_style='success')
btn_download = widgets.Button(description="Download Result", button_style='info', disabled=True)
out_display = widgets.Output()

TEMP_OUTPUT_PATH = "retro_rebirth_result.jpg"

def process_retro_image(b):
    out_display.clear_output()
    btn_download.disabled = True

    if not btn_upload.value:
        with out_display:
            print("Please upload an image first.")
        return

    # Load upload
    fname = list(btn_upload.value.keys())[0]
    content = btn_upload.value[fname]['content']
    input_pil = PILImage.open(io.BytesIO(content)).convert("RGB")

    with out_display:
        print(f"Processing as {dropdown_mode.value}...")

        # -----------------------------------------
        # MODE 1 → NEGATIVE RESTORATION (Your U-Net)
        # -----------------------------------------
        if dropdown_mode.value == 'Negative':
            with torch.no_grad():
                inp = input_pil.resize((256, 256))
                t_in = transforms.ToTensor()(inp).unsqueeze(0).to(device)
                res = model(t_in).squeeze(0).cpu().permute(1, 2, 0).numpy()

                result_pil = PILImage.fromarray((res * 255).astype(np.uint8)).resize(input_pil.size)

        # -----------------------------------------
        # MODE 2 → B&W COLORIZATION (DDColor)
        # -----------------------------------------
        else:
            input_pil.save("temp_bw.jpg")
            result = color_pipeline("temp_bw.jpg")
            result_pil = PILImage.fromarray(
                cv2.cvtColor(result['output_img'], cv2.COLOR_BGR2RGB)
            )

        # Save and show output
        result_pil.save(TEMP_OUTPUT_PATH)
        btn_download.disabled = False

        plt.figure(figsize=(12,6))
        plt.subplot(1,2,1); plt.imshow(input_pil); plt.axis('off'); plt.title("Original Input")
        plt.subplot(1,2,2); plt.imshow(result_pil); plt.axis('off'); plt.title("Result")
        plt.show()

def download_image(b):
    with out_display:
        if os.path.exists(TEMP_OUTPUT_PATH):
            from google.colab import files
            files.download(TEMP_OUTPUT_PATH)
        else:
            print("No image generated yet.")

btn_run.on_click(process_retro_image)
btn_download.on_click(download_image)

display(widgets.VBox([
    widgets.Label("Step 1: Upload Image"),
    btn_upload,
    widgets.Label("Step 2: Choose Mode"),
    dropdown_mode,
    widgets.HBox([btn_run, btn_download]),
    out_display
]))


🎉 Loaded trained RetroRebirth U-Net successfully!
⏳ Loading DDColor...


2025-12-08 12:45:40,309 - modelscope - WARNING - Model revision not specified, use revision: v1.02
2025-12-08 12:45:40,559 - modelscope - INFO - initiate model from /root/.cache/modelscope/hub/damo/cv_ddcolor_image-colorization
2025-12-08 12:45:40,560 - modelscope - INFO - initiate model from location /root/.cache/modelscope/hub/damo/cv_ddcolor_image-colorization.
2025-12-08 12:45:40,563 - modelscope - INFO - initialize model from /root/.cache/modelscope/hub/damo/cv_ddcolor_image-colorization
2025-12-08 12:45:48,680 - modelscope - INFO - Loading DDColor model from /root/.cache/modelscope/hub/damo/cv_ddcolor_image-colorization/pytorch_model.pt, with param key: [params].
2025-12-08 12:45:49,364 - modelscope - INFO - load model done.
2025-12-08 12:45:49,399 - modelscope - WARNING - No preprocessor field found in cfg.
2025-12-08 12:45:49,399 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-12-08 12:45:49,400 - modelscope - WARNI

✅ DDColor Ready!
